# Bài 24: SEC EDGAR auto-fetch
## Gõ ticker → tự kéo 10-Q mới nhất về, không cần upload

**Mục tiêu:**
- Dùng API miễn phí của SEC EDGAR: ticker → CIK → 10-Q mới nhất
- Bóc text từ HTML báo cáo bằng BeautifulSoup
- Refactor `ingest.py`: tách lõi `ingest_text()` dùng chung cho cả PDF lẫn HTML
- Nút 'Kéo 10-Q từ SEC' trong UI

**Vì sao là tính năng 'wow':** đến giờ user vẫn phải tự có file PDF. Với EDGAR, họ chỉ gõ `TSLA` là có ngay báo cáo mới nhất — xóa hẳn khoảng trống dữ liệu.

---
## Phần 1: Lý thuyết

### SEC EDGAR API — miễn phí, không cần key

SEC bắt buộc mọi request có header `User-Agent` ghi rõ danh tính (tên + email), nếu không sẽ bị chặn:
```python
SEC_HEADERS = {"User-Agent": "StockResearchAssistant your_email@gmail.com"}
```

### 3 bước lấy 10-Q

```
1. ticker → CIK        GET sec.gov/files/company_tickers.json
                       (bảng tra: {"ticker": "AAPL", "cik_str": 320193, ...})

2. CIK → filings       GET data.sec.gov/submissions/CIK{cik10}.json
                       (filings["recent"] có các mảng SONG SONG:
                        form[], accessionNumber[], primaryDocument[], filingDate[])

3. → tải HTML 10-Q     GET sec.gov/Archives/edgar/data/{cik}/{accession}/{doc}
```

- **CIK** = mã định danh công ty ở SEC. Cần pad 0 thành 10 chữ số cho URL submissions (`320193` → `0000320193`).
- Các mảng trong `recent` **xếp mới nhất trước** → 10-Q đầu tiên tìm thấy chính là bản mới nhất.

---

### HTML ≠ PDF → cần bóc text

10-Q trên EDGAR là **HTML**, không phải PDF. Nên `ingest_pdf` (đọc bytes qua fitz) không dùng được. Ta bóc text bằng BeautifulSoup:
```python
from bs4 import BeautifulSoup
text = BeautifulSoup(html, "html.parser").get_text(separator=" ", strip=True)
```

---

### Refactor: một lõi ingest dùng chung

Giờ có 2 nguồn (PDF upload, HTML EDGAR) nhưng phần *chunk → embed → lưu* **giống hệt nhau**. Đừng viết 2 lần — tách thành lõi chung:

```
ingest_pdf(bytes)  ─┐
                    ├─→  ingest_text(full_text, symbol, source)  →  ChromaDB
edgar → html→text ─┘         (chunk + embed + upsert idempotent)
```

`ingest_pdf` chỉ còn lo *đọc PDF → text*, rồi gọi `ingest_text`. Nguồn EDGAR cũng chỉ cần *lấy text* rồi gọi `ingest_text`. DRY.

---
## Phần 2: Ví dụ — kéo 10-Q của một công ty (chạy thật)

Ô dưới gọi API SEC thật: ticker → CIK → 10-Q mới nhất → bóc text. Đổi `TICKER` để thử công ty khác.

In [1]:
import requests
from bs4 import BeautifulSoup

SEC_HEADERS = {"User-Agent": "StockResearchAssistant hhoang181004@gmail.com"}
TICKER = "AAPL"

# Bước 1: ticker → CIK
tickers = requests.get("https://www.sec.gov/files/company_tickers.json", headers=SEC_HEADERS).json()
cik = None
for entry in tickers.values():
    if entry["ticker"] == TICKER:
        cik = str(entry["cik_str"]).zfill(10)   # pad 0 → 10 chữ số
        break
print(f"{TICKER} → CIK {cik}")

# Bước 2: CIK → tìm 10-Q mới nhất
subs = requests.get(f"https://data.sec.gov/submissions/CIK{cik}.json", headers=SEC_HEADERS).json()
recent = subs["filings"]["recent"]
doc_url, filing_date = None, None
for form, acc, doc, date in zip(
    recent["form"], recent["accessionNumber"], recent["primaryDocument"], recent["filingDate"]
):
    if form == "10-Q":
        acc_nodash = acc.replace("-", "")
        doc_url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_nodash}/{doc}"
        filing_date = date
        break
print(f"10-Q mới nhất ({filing_date}): {doc_url}")

# Bước 3: tải HTML → bóc text
html = requests.get(doc_url, headers=SEC_HEADERS).text
text = BeautifulSoup(html, "html.parser").get_text(separator=" ", strip=True)
print(f"\nĐộ dài text: {len(text):,} ký tự")
print("500 ký tự đầu:\n", text[:500])

AAPL → CIK 0000320193
10-Q mới nhất (2026-05-01): https://www.sec.gov/Archives/edgar/data/320193/000032019326000013/aapl-20260328.htm

Độ dài text: 96,751 ký tự
500 ký tự đầu:
 aapl-20260328 false 2026 Q2 0000320193 --09-26 P1Y P1Y P1Y P1Y http://fasb.org/us-gaap/2025#LongTermDebtCurrent http://fasb.org/us-gaap/2025#LongTermDebtNoncurrent http://fasb.org/us-gaap/2025#LongTermDebtCurrent http://fasb.org/us-gaap/2025#LongTermDebtNoncurrent P329D xbrli:shares iso4217:USD iso4217:USD xbrli:shares xbrli:pure aapl:Customer aapl:Vendor 0000320193 2025-09-28 2026-03-28 0000320193 us-gaap:CommonStockMember 2025-09-28 2026-03-28 0000320193 aapl:A1.625NotesDue2026Member 2025-09-2


Chạy được → ta có text 10-Q mới nhất chỉ từ 1 ticker. Giờ đóng gói vào app.

---
## Phần 3: Bài tập

### Bước 1: Refactor `app/services/ingest.py` — tách lõi `ingest_text()`

Tách phần *chunk → embed → lưu* của `ingest_pdf` thành hàm riêng `ingest_text`. Sau đó `ingest_pdf` chỉ đọc PDF → text rồi gọi `ingest_text`.

```python
def ingest_text(full_text: str, symbol: str, source_name: str) -> int:
    """Lõi chung: text → chunk → embed → lưu ChromaDB (idempotent).
    Dùng cho MỌI nguồn: PDF upload, HTML từ EDGAR..."""
    logger.info(f"[ingest] Bắt đầu ingest '{source_name}' cho symbol {symbol}")
    chunks = split_into_chunks(full_text)
    if not chunks:
        logger.warning(f"[ingest] '{source_name}' không có text — bỏ qua")
        return 0
    embed_model, collection = get_resources()
    collection.delete(where={"$and": [{"symbol": symbol}, {"source": source_name}]})
    embeddings = embed_model.encode(chunks).tolist()
    metadatas = [{"symbol": symbol, "source": source_name} for _ in chunks]
    ids = [f"{symbol}::{source_name}::{i}" for i in range(len(chunks))]
    collection.add(documents=chunks, embeddings=embeddings, metadatas=metadatas, ids=ids)
    logger.info(f"[ingest] Đã thêm {len(chunks)} chunks từ '{source_name}'")
    return len(chunks)


def ingest_pdf(file_bytes: bytes, symbol: str, source_name: str) -> int:
    """PDF bytes → text → ingest_text."""
    doc = fitz.open(stream=file_bytes, filetype="pdf")
    full_text = ""
    for page_num in range(doc.page_count):
        full_text += doc[page_num].get_text()
    doc.close()
    # TODO: gọi ingest_text(...) và return kết quả
```

`detect_symbol_from_pdf` giữ nguyên.

---

### Bước 2: Tạo `app/services/edgar.py`

Điền `TODO` (tái dùng đúng logic Phần 2):

```python
import requests
from bs4 import BeautifulSoup
from logging_setup import logger

SEC_HEADERS = {"User-Agent": "StockResearchAssistant hhoang181004@gmail.com"}


def get_cik(ticker: str) -> str | None:
    """ticker → CIK (10 chữ số). None nếu không tìm thấy."""
    tickers = requests.get("https://www.sec.gov/files/company_tickers.json", headers=SEC_HEADERS).json()
    ticker = ticker.upper()
    for entry in tickers.values():
        if entry["ticker"] == ticker:
            return str(entry["cik_str"]).zfill(10)
    return None


def fetch_latest_10q(ticker: str) -> tuple[str, str] | None:
    """Trả về (text, source_name) của 10-Q mới nhất. None nếu không có."""
    cik = get_cik(ticker)
    if not cik:
        logger.warning(f"[edgar] Không tìm thấy CIK cho {ticker}")
        return None

    subs = requests.get(f"https://data.sec.gov/submissions/CIK{cik}.json", headers=SEC_HEADERS).json()
    recent = subs["filings"]["recent"]

    for form, acc, doc, date in zip(
        recent["form"], recent["accessionNumber"], recent["primaryDocument"], recent["filingDate"]
    ):
        if form == "10-Q":
            acc_nodash = acc.replace("-", "")
            # TODO: dựng doc_url (xem Phần 2), tải html, bóc text bằng BeautifulSoup
            #   doc_url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_nodash}/{doc}"
            #   html = requests.get(doc_url, headers=SEC_HEADERS).text
            #   text = BeautifulSoup(html, "html.parser").get_text(separator=" ", strip=True)
            source_name = f"{ticker.upper()}-10Q-{date}-SEC"
            logger.info(f"[edgar] Lấy 10-Q {ticker} ({date})")
            # TODO: return (text, source_name)

    logger.warning(f"[edgar] {ticker} không có 10-Q")
    return None
```

> Nhớ đổi email trong `SEC_HEADERS` thành email của bạn (SEC khuyến nghị vậy).

---

### Bước 3: Nút 'Kéo 10-Q từ SEC' trong `app/main.py`

Thêm import: `from services.edgar import fetch_latest_10q` và (nếu chưa) `from services.ingest import ingest_text`.

Thêm khối này trong sidebar (dưới phần upload, trên mục Kho tài liệu):

```python
    st.markdown("---")
    st.subheader("🏛️ Kéo báo cáo từ SEC")
    edgar_ticker = st.text_input("Mã cổ phiếu (Mỹ)", key="edgar_ticker", placeholder="VD: TSLA")
    if st.button("⬇️ Kéo 10-Q mới nhất"):
        ticker = edgar_ticker.upper().strip()
        if not ticker:
            st.warning("Nhập mã cổ phiếu trước.")
        else:
            with st.spinner(f"Đang lấy 10-Q của {ticker} từ SEC..."):
                result = fetch_latest_10q(ticker)
            # TODO: nếu result is None → st.error("Không tìm thấy 10-Q")
            # Nếu có → text, source = result
            #   n = ingest_text(text, ticker, source)
            #   st.success(f"Đã thêm {n} chunks từ {source}")
            #   st.rerun()   # cập nhật mục Kho tài liệu
```

---

### Bước 4: `requirements.txt` + Test

Thêm vào `requirements.txt` (đã cài sẵn trong máy, nhưng ghi vào để tái lập được):
```
# Phase 5 - SEC EDGAR
beautifulsoup4==4.15.0
```

Chạy `streamlit run app/main.py`:
1. Mục '🏛️ Kéo báo cáo từ SEC', gõ `TSLA` → bấm Kéo
2. Log hiện `[edgar] Lấy 10-Q TSLA (...)` rồi `[ingest] Đã thêm N chunks`
3. Mục 📚 Kho tài liệu hiện `TSLA — 1 tài liệu` (nguồn `TSLA-10Q-...-SEC`)
4. Hỏi "Doanh thu TSLA quý gần nhất?" → trả lời dựa trên 10-Q vừa kéo, citations chỉ đúng nguồn SEC

**Checklist:**
- [ ] Gõ 1 ticker là có báo cáo, không cần upload
- [ ] Kéo lại cùng ticker → không nhân đôi (idempotent qua `ingest_text`)
- [ ] Ticker không có 10-Q (vd công ty nước ngoài nộp 20-F) → báo lỗi gọn, không crash